In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
import sys

sys.path.append("..")

In [2]:
DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
PLOTS_DIR = "../plots"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/sig_with_layer_pixel_spacetime.npy"
BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/bg_with_layer_pixel_spacetime.npy"
SIGNAL_MPPC_FILE = f"{DATA_DIR}/sig_with_layer_mppc_spacetime.npy"
BACKGROUND_MPPC_FILE = f"{DATA_DIR}/bg_with_layer_mppc_spacetime.npy"
SIGNAL_ONLY_PIXEL_FILE = f"{DATA_DIR}/sig_only_with_layer_pixel_spacetime.npy"
SIGNAL_ONLY_MPPC_FILE = f"{DATA_DIR}/sig_only_with_layer_mppc_spacetime.npy"

bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)

from src.utils import get_spacetime_data
bg_pixel_spacetime = get_spacetime_data(bg_pixel_spacetime)
sig_pixel_spacetime = get_spacetime_data(sig_pixel_spacetime)


bg_pixel_seqlen = (bg_pixel_spacetime != -1).all(axis=-1).sum(axis=-1)
sig_pixel_seqlen = (sig_pixel_spacetime != -1).all(axis=-1).sum(axis=-1)

input_seq_len = bg_pixel_spacetime.shape[1]
input_dim = bg_pixel_spacetime.shape[2]  # Exclude timestamp

from src.utils import cartesian_to_cylindrical



bg_pixel_cylindrical = np.concatenate(
    [
        cartesian_to_cylindrical(bg_pixel_spacetime[:, :, :2]),
        bg_pixel_spacetime[:, :, 2:],
    ],
    axis=-1,
)
sig_pixel_cylindrical = np.concatenate(
    [
        cartesian_to_cylindrical(sig_pixel_spacetime[:, :, :2]),
        sig_pixel_spacetime[:, :, 2:],
    ],
    axis=-1,
)

bg_seq_length = (bg_pixel_spacetime != -1).all(axis=-1).sum(axis=-1)
sig_seq_length = (sig_pixel_spacetime != -1).all(axis=-1).sum(axis=-1)

bg_pixel_spacetime = bg_pixel_spacetime[bg_pixel_seqlen > 0]
sig_pixel_spacetime = sig_pixel_spacetime[sig_pixel_seqlen > 0]
bg_pixel_cylindrical = bg_pixel_cylindrical[bg_pixel_seqlen > 0]
sig_pixel_cylindrical = sig_pixel_cylindrical[sig_pixel_seqlen > 0]
bg_seq_length = bg_seq_length[bg_pixel_seqlen > 0]
sig_seq_length = sig_seq_length[sig_pixel_seqlen > 0]

In [6]:
from src.keras.model.components import (
    MultiHeadAttentionBlock,
    MultiHeadAttentionStack,
    SelfAttentionBlock,
    SelfAttentionStack,
    PoolingAttentionBlock,
    GenerateMask,
    GetSequenceLength,
    DecoderQueries,
    GenerateDecoderMask,
    MLP,
    MaskOutput,
)

feature_dim = 16
latent_dim = 64
num_seeds = latent_dim // feature_dim
dropout_rate = 0.0
num_heads = 8

pixel_input = keras.layers.Input(shape=(input_seq_len, input_dim), name="pixel_input")
pixel_mask = GenerateMask(name="pixel_mask")(pixel_input)
pixel_seqlen = GetSequenceLength(name="pixel_seqlen")(pixel_mask)

pixel_embedding = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="pixel_embedding",
)(pixel_input)

pixel_self_attention = SelfAttentionStack(
    num_heads=num_heads,
    key_dim=feature_dim,
    stack_size=3,
    name="pixel_self_attention",
)(pixel_embedding, pixel_mask)

pixel_pooling = PoolingAttentionBlock(
    key_dim=feature_dim,
    num_seeds=num_seeds,
    num_heads=num_heads,
    name="pixel_pooling",
)(pixel_self_attention, pixel_mask)

pixel_latent_space = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="pixel_latent_space",
)(pixel_pooling)


pixel_latent_output = keras.layers.Flatten(name="pixel_latent_output")(
    pixel_latent_space
)


decoder_queries = DecoderQueries(
    num_queries=input_seq_len,
    feature_dim=feature_dim,
    name="decoder_queries",
)(pixel_seqlen)

decoded_embedded_queries = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="decoded_embedded_queries",
)(decoder_queries)


decoder_mask = GenerateDecoderMask(name="decoder_mask", max_length=input_seq_len)(pixel_seqlen)

decoded_latent_set = MLP(
    num_layers=4,
    output_dim=feature_dim,
    name="decoded_latent_set",
)(pixel_latent_space)


decoded_pixel_space = MultiHeadAttentionStack(
    num_heads=num_heads,
    key_dim=feature_dim,
    stack_size=3,
    name="decoded_pixel_space",
)(decoder_queries, decoded_latent_set, query_mask=decoder_mask)


decoded_point_set_self_attention = SelfAttentionStack(
    num_heads=num_heads,
    key_dim=feature_dim,
    stack_size=3,
    name="decoded_point_set_self_attention",
)(decoded_pixel_space, decoder_mask)

decoded_point_set = MLP(
    num_layers=4,
    output_dim=input_dim,
    name="decoded_point_set_mlp",
    activation="linear",
)(decoded_point_set_self_attention)

decoded_output = MaskOutput(name="decoded_output", padding_value=-1)(
    decoded_point_set, decoder_mask
)

"""
seqlen_output = keras.layers.Concatenate(name="seqlen_output")(
    [
        keras.layers.Flatten(name="seqlen_flatten")(pixel_seqlen),
        keras.layers.Flatten(name="decoder_seqlen_flatten")(decoder_seqlen),
    ]
)"""

model = keras.Model(
    inputs=pixel_input,
    outputs=[decoded_output, pixel_latent_output],
    name="SetAutoEncoder",
)

from src.keras.training import SplitMSE, ChamferDistanceMasked, EmbeddingSpaceSpreading

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={
        "decoded_output": ChamferDistanceMasked(padding_value=-1),
        "pixel_latent_output": EmbeddingSpaceSpreading(),
    },
)

model.summary()

Model: "SetAutoEncoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ pixel_input         │ (None, 256, 4)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_mask          │ (None, 256, 1)    │          0 │ pixel_input[0][0] │
│ (GenerateMask)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_embedding     │ (None, 256, 16)   │        364 │ pixel_input[0][0] │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_self_attenti… │ (None, 256, 16)   │     29,184 │ pixel_embedding[… │
│ (SelfAttentionStac… │                   │            │ pixel_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_pooling       │ (None, 4, 16)     │     10,896 │ pixel_self_atten… │
│ (PoolingAttentionB… │                   │            │ pixel_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_seqlen        │ (None, 1)         │          0 │ pixel_mask[0][0]  │
│ (GetSequenceLength) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_latent_space  │ (None, 4, 16)     │      1,088 │ pixel_pooling[0]… │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_queries     │ (None, 256, 16)   │      4,096 │ pixel_seqlen[0][… │
│ (DecoderQueries)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_latent_set  │ (None, 4, 16)     │      1,088 │ pixel_latent_spa… │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_mask        │ (None, 256, 1)    │          0 │ pixel_seqlen[0][… │
│ (GenerateDecoderMa… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_pixel_space │ (None, 256, 16)   │     29,280 │ decoder_queries[… │
│ (MultiHeadAttentio… │                   │            │ decoded_latent_s… │
│                     │                   │            │ decoder_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_point_set_… │ (None, 256, 16)   │     29,184 │ decoded_pixel_sp… │
│ (SelfAttentionStac… │                   │            │ decoder_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_point_set_… │ (None, 256, 4)    │        352 │ decoded_point_se… │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoded_output      │ (None, 256, 4)    │          0 │ decoded_point_se… │
│ (MaskOutput)        │                   │            │ decoder_mask[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_latent_output │ (None, 64)        │          0 │ pixel_latent_spa… │
│ (Flatten)           │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 105,532 (412.23 KB)

 Trainable params: 105,532 (412.23 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
data = bg_pixel_spacetime[:20000]
model.fit(
    x=data,
    y={
        "decoded_output": data,
        "pixel_latent_output": np.zeros((data.shape[0], latent_dim)),
    },
    batch_size=512,
    epochs=100,
    validation_split=0.1,
    shuffle=True,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
        ),
        keras.callbacks.LearningRateScheduler(
            lambda epoch: 1e-3 * 0.9 ** (epoch // 5),
            verbose=1,
        )
    ],
)


Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/100


/Users/simi/mu3e_trigger/venv/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'self_attention_block_12' (of type SelfAttentionBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/Users/simi/mu3e_trigger/venv/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'self_attention_block_13' (of type SelfAttentionBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/Users/simi/mu3e_trigger/venv/lib/python3.10/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'self_attention_block_14' (of type SelfAttentionBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will theref

 2/36 ━━━━━━━━━━━━━━━━━━━━ 5:32 10s/step - decoded_output_loss: 11137.9336 - loss: 11162.3721 - pixel_latent_output_loss: 24.4379 

KeyboardInterrupt: 

In [ ]:
pred = model.predict(x=sig_pixel_cylindrical[:1], batch_size=10)

In [ ]:
encoder = keras.Model(
    inputs=model.inputs,
    outputs=pixel_latent_output,
    name="SetAutoEncoderEncoder",
)

encoder.predict(x=sig_pixel_spacetime[:2], batch_size=10)

In [ ]:
event = sig_pixel_spacetime[1:2]
pred = model.predict(x=event, batch_size=10)
plt.scatter(
    pred[0][0, :, 0],
    pred[0][0, :, 1],
    c=pred[0][0, :, 2],
    s=10,
    marker="o",
)
plt.scatter(
    event[0, :, 0],
    event[0, :, 1],
    c=event[0, :, 2],
    s=10,
    marker="x",
)